In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
%sql
CREATE WIDGET TEXT catalog_name DEFAULT 'smartdata_project';
CREATE WIDGET TEXT schema_source DEFAULT 'silver';
CREATE WIDGET TEXT schema_sink DEFAULT 'gold';
CREATE WIDGET TEXT source_products DEFAULT 'silver_products';
CREATE WIDGET TEXT sink_products DEFAULT 'gold_products';
CREATE WIDGET TEXT w_products DEFAULT '1900-01-01';

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_source = dbutils.widgets.get('schema_source')
schema_sink = dbutils.widgets.get('schema_sink')
source_products = dbutils.widgets.get('source_products')
sink_products = dbutils.widgets.get('sink_products')
w_products = dbutils.widgets.get('w_products')

In [0]:
source_products_table = f'{catalog_name}.{schema_source}.{source_products}'
df_sales = spark.sql(f"SELECT * FROM {source_products_table} WHERE updated_at > '{w_products}'")

In [0]:
df_gold_products = (
    df_sales
    .withColumn('class_price', 
        F.when(F.col('price') >= 500, 'premium')
        .when(F.col('price') >= 200, 'average').otherwise('cheap')
    )
)

In [0]:
df_gold_products.printSchema()

In [0]:
target_sink_table = f'{catalog_name}.{schema_sink}.{sink_products}'
if spark.catalog.tableExists(target_sink_table):

    source_table = DeltaTable.forName(spark, target_sink_table)

    (
        source_table.alias("target")
        .merge(
            df_gold_products.alias("source"),
            f"target.product_id = source.product_id"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df_gold_products.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('product_id','updated_at','price')  # Liquid Clustering
        .saveAsTable(target_sink_table)
    )